# Exploratory Data Analysis: Student Performance

This notebook uses reproducible synthetic data. Patterns are intentionally generated and should not be interpreted as evidence about real students.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
data_path = ROOT / 'data' / 'student_data.csv'
if not data_path.exists():
    from src.generate_data import make_dataset
    df = make_dataset()
else:
    df = pd.read_csv(data_path)
print(f'Raw shape: {df.shape}; duplicate rows: {df.duplicated().sum()}')
df.head()

## Clean and inspect
Exact duplicate rows are removed. Missing feature values remain for now so plots can skip them and the training pipeline can impute them consistently.

In [ ]:
df = df.drop_duplicates().copy()
print(df.isna().sum())
df.describe(include='all').T

## Score distribution and study hours
The scatter plot can show the synthetic association; it does not establish causation.

In [ ]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df, x='academic_score', bins=25, ax=axes[0], color='#5470c6')
sns.regplot(data=df, x='study_hours_per_day', y='academic_score',
            scatter_kws={'alpha': .35}, line_kws={'color': '#d62728'}, ax=axes[1])
axes[0].set_title('Synthetic score distribution')
axes[1].set_title('Study time and synthetic score')
plt.tight_layout()

## Attendance versus performance
Each point represents one generated record. The fitted line is a visual summary, not a causal claim.

In [ ]:
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df, x='attendance_percentage', y='academic_score', alpha=.45)
sns.regplot(data=df, x='attendance_percentage', y='academic_score', scatter=False, color='#d62728')
plt.title('Attendance and synthetic academic score')
plt.tight_layout()

## Feature correlations
Correlation measures linear co-movement. It does not prove one factor causes another.

In [ ]:
numeric = df.select_dtypes(include='number')
plt.figure(figsize=(9, 6))
sns.heatmap(numeric.corr(), annot=True, fmt='.2f', cmap='vlag', center=0)
plt.title('Numeric feature correlation matrix')
plt.tight_layout()

## Next steps
Continue with `python src/train_model.py` to compare Linear Regression and Random Forest on a held-out split. Metrics apply only to this synthetic dataset.